In [1]:
!pip install pandas pyarrow numpy matplotlib seaborn -q


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


Load our enriched dataset

In [4]:
df = pd.read_parquet(
    "smard_1year_enriched.parquet"
)

print("Dataset loaded!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded!
Shape: (8619, 14)


,timestamp,consumption_mw,wind_onshore_mw,solar_mw,gas_mw,coal_mw,price_eur_mwh,wind_offshore_mw,biomass_mw,hydro_mw,lignite_mw,other_renewables_mw,renewable_generation_mw,renewable_share_pct
0,2025-08-31 22:00:00,39947.01,12740.28,7.38,4596.50,1664.75,84.08,4774.56,3552.27,1632.67,7031.50,100.30,22807.46,57.094286
1,2025-08-31 23:00:00,39017.77,13182.18,7.49,4538.50,1616.25,81.43,3709.63,3478.34,1622.75,6791.75,100.94,22101.33,56.644267
2,2025-09-01 00:00:00,37956.16,12593.54,8.50,4705.75,1627.25,80.97,2375.40,3455.61,1632.35,6920.00,101.41,20166.81,53.131850
3,2025-09-01 01:00:00,37650.69,12599.03,7.91,4701.50,1632.00,80.03,1490.00,3467.22,1545.54,7185.50,101.57,19211.27,51.025014
4,2025-09-01 02:00:00,38324.95,12846.25,7.47,4962.75,1725.00,81.37,765.38,3537.74,1556.14,7308.50,101.49,18814.47,49.091962


In [5]:
print("Columns:")
for column in df.columns:
    print("-", column)

Columns:
- timestamp
- consumption_mw
- wind_onshore_mw
- solar_mw
- gas_mw
- coal_mw
- price_eur_mwh
- wind_offshore_mw
- biomass_mw
- hydro_mw
- lignite_mw
- other_renewables_mw
- renewable_generation_mw
- renewable_share_pct


Create time features

In [6]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["month"] = df["timestamp"].dt.month
df["month_name"] = df["timestamp"].dt.month_name()

print("Time features created!")

display(
    df[
        [
            "timestamp",
            "hour",
            "day_of_week",
            "month_name"
        ]
    ].head()
)

Time features created!


,timestamp,hour,day_of_week,month_name
0,2025-08-31 22:00:00,22,Sunday,August
1,2025-08-31 23:00:00,23,Sunday,August
2,2025-09-01 00:00:00,0,Monday,September
3,2025-09-01 01:00:00,1,Monday,September
4,2025-09-01 02:00:00,2,Monday,September


# **Basic KPIs**

In [7]:
average_price = df["price_eur_mwh"].mean()
average_consumption = df["consumption_mw"].mean()
peak_consumption = df["consumption_mw"].max()
average_renewable_share = df["renewable_share_pct"].mean()

print("========== GRID KPIs ==========")
print(f"Average electricity price: €{average_price:.2f}/MWh")
print(f"Average consumption: {average_consumption:,.0f} MW")
print(f"Peak consumption: {peak_consumption:,.0f} MW")
print(f"Average renewable share: {average_renewable_share:.1f}%")
print("================================")

========== GRID KPIs ==========
Average electricity price: €98.74/MWh
Average consumption: 53,853 MW
Peak consumption: 78,241 MW
Average renewable share: 58.6%


Negative-price analysis

In [8]:
negative_prices = df[
    df["price_eur_mwh"] < 0
]

negative_hours = len(negative_prices)

percentage_negative = (
    negative_hours / len(df) * 100
)

print("====== NEGATIVE PRICE ANALYSIS ======")
print("Negative-price hours:", negative_hours)
print(f"Percentage of hours: {percentage_negative:.2f}%")
print(
    f"Lowest price: "
    f"€{df['price_eur_mwh'].min():.2f}/MWh"
)
print("======================================")

====== NEGATIVE PRICE ANALYSIS ======
Negative-price hours: 514
Percentage of hours: 5.96%
Lowest price: €-499.00/MWh


Daily statistics

In [9]:
daily = (
    df.groupby("date")
    .agg(
        average_price=("price_eur_mwh", "mean"),
        minimum_price=("price_eur_mwh", "min"),
        maximum_price=("price_eur_mwh", "max"),
        average_consumption=("consumption_mw", "mean"),
        peak_consumption=("consumption_mw", "max"),
        average_renewable_share=("renewable_share_pct", "mean")
    )
    .reset_index()
)

print("Daily dataset shape:", daily.shape)

display(daily.head())

Daily dataset shape: (361, 7)


,date,average_price,minimum_price,maximum_price,average_consumption,peak_consumption,average_renewable_share
0,2025-08-31,82.755000,81.43,84.08,39482.390000,39947.01,56.869277
1,2025-09-01,102.278333,25.03,250.65,51529.228750,59875.09,44.192183
2,2025-09-02,122.207500,78.54,311.51,52054.538750,60391.85,44.577208
3,2025-09-03,59.479167,-0.28,112.34,53512.738750,60666.30,78.145835
4,2025-09-04,93.528333,3.20,350.00,53009.459583,60435.25,59.243001


Renewable vs price correlation

In [10]:
correlation = df[
    [
        "renewable_generation_mw",
        "renewable_share_pct",
        "price_eur_mwh"
    ]
].corr()

print("Correlation matrix:")

display(correlation)

Correlation matrix:


,renewable_generation_mw,renewable_share_pct,price_eur_mwh
renewable_generation_mw,1.000000,0.931196,-0.700960
renewable_share_pct,0.931196,1.000000,-0.783068
price_eur_mwh,-0.700960,-0.783068,1.000000


In [11]:
daily.to_parquet(
    "daily_analytics.parquet",
    index=False
)

df.to_parquet(
    "final_analytics_data.parquet",
    index=False
)

print("Analytics files saved!")
print()
print("Created:")
print("- daily_analytics.parquet")
print("- final_analytics_data.parquet")

Analytics files saved!

Created:
- daily_analytics.parquet
- final_analytics_data.parquet
